In [ ]:
import numpy as np
from skimage.util import view_as_windows
import matplotlib.pyplot as plt

from utils_experiments import scale_to_range, unscale_from_range

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


In [ ]:
seis_mature= np.load('../2024_tesis_maestria/data/data_decatur/processed/seismic_mature_block.npy')
phi_mature = np.load('../2024_tesis_maestria/data/data_decatur/processed/porosity_mature_block.npy')

seis_exploration = np.load('../2024_tesis_maestria/data/data_decatur/processed/seismic_exploration_block.npy')
phi_exploration = np.load('../2024_tesis_maestria/data/data_decatur/processed/porosity_exploration_block.npy')

phi_mature[phi_mature<0] = 0
phi_exploration[phi_exploration<0] = 0


In [ ]:
seis_mature = scale_to_range(seis_mature)
phi_mature = scale_to_range(phi_mature)

seis_exploration = scale_to_range(seis_exploration)
phi_exploration = scale_to_range(phi_exploration)


In [ ]:
patch_spec = {'patch_size': (16, 16), 'step': 16}

In [ ]:
def create_2d_patches(
    data_3d: np.ndarray,
    patch_size: tuple = (16, 16),
    step: int = 16,
    axis_to_slice: int = 2
) -> np.ndarray:
    """
    Slices a 3D NumPy array into 2D patches.

    Args:
        data_3d (np.ndarray): The 3D seismic volume. Assumes shape is (dim1, dim2, dim3).
        patch_size (tuple): The (height, width) of the patches. Defaults to (16, 16).
        step (int): The step size for the sliding window. A step equal to the 
                    patch size creates non-overlapping patches. Defaults to 16.
        axis_to_slice (int): The axis to iterate over for creating 2D slices. 
                             0 for inline, 1 for crossline, 2 for depth/time. 
                             Defaults to 2.

    Returns:
        np.ndarray: A 4D array of patches with shape 
                    (num_patches, patch_height, patch_width, 1).
    """
    if data_3d.ndim != 3:
        raise ValueError("Input data must be a 3D array.")
    if axis_to_slice < 0 or axis_to_slice > 2:
        raise ValueError("axis_to_slice must be 0, 1, or 2.")

    all_patches = []
    
    # Move the slicing axis to the first position for easier iteration
    data_3d = np.moveaxis(data_3d, axis_to_slice, 0)

    # Loop through each 2D slice
    for slice_2d in data_3d:
        # Use view_as_windows to extract patches
        patches_in_slice = view_as_windows(slice_2d, window_shape=patch_size, step=step)
        
        # Reshape to a list of patches: (num_patches_in_slice, height, width)
        h, w, ph, pw = patches_in_slice.shape
        patches_in_slice = patches_in_slice.reshape(h * w, ph, pw)
        
        if patches_in_slice.size > 0:
            all_patches.append(patches_in_slice)

    if not all_patches:
        return np.array([]) # Return empty if no patches could be created

    # Combine all patches into a single NumPy array
    final_patches_array = np.concatenate(all_patches, axis=0)

    # Add a channel dimension for CNN input: (num_patches, height, width, 1)
    return np.expand_dims(final_patches_array, axis=-1)

In [ ]:
# 2. Generate patches from INLINE view (slicing along axis 0)
# Each 2D slice will have shape (crossline, depth) -> (120, 80)
seis_mature_inline_patches = create_2d_patches(seis_mature, **patch_spec, axis_to_slice=0)
print(f"Generated {seis_mature_inline_patches.shape[0]} patches from the INLINE view.")

# 3. Generate patches from CROSSLINE view (slicing along axis 1)
# Each 2D slice will have shape (inline, depth) -> (100, 80)
seis_mature_crossline_patches = create_2d_patches(seis_mature, **patch_spec, axis_to_slice=1)
print(f"Generated {seis_mature_crossline_patches.shape[0]} patches from the CROSSLINE view.")

# 4. Concatenate the two sets of patches into one large dataset
seis_mature_patches = np.concatenate([seis_mature_inline_patches, seis_mature_crossline_patches], axis=0)

print(f"\nOriginal data shape: {seis_mature.shape}")
print(f"Total combined patches for the CNN: {seis_mature_patches.shape}")

In [ ]:
# Seis exploration
seis_exploration_inline_patches = create_2d_patches(seis_exploration, **patch_spec, axis_to_slice=0)
seis_exploration_crossline_patches = create_2d_patches(seis_exploration, **patch_spec, axis_to_slice=1)
seis_exploration_patches = np.concatenate([seis_exploration_inline_patches, seis_exploration_crossline_patches], axis=0)

#Phi exploration
phi_exploration_inline_patches = create_2d_patches(phi_exploration, **patch_spec, axis_to_slice=0)
phi_exploration_crossline_patches = create_2d_patches(phi_exploration, **patch_spec, axis_to_slice=1)
phi_exploration_patches = np.concatenate([phi_exploration_inline_patches, phi_exploration_crossline_patches], axis=0)

#Phi mature
phi_mature_inline_patches = create_2d_patches(phi_mature, **patch_spec, axis_to_slice=0)
phi_mature_crossline_patches = create_2d_patches(phi_mature, **patch_spec, axis_to_slice=1)
phi_mature_patches = np.concatenate([phi_mature_inline_patches, phi_mature_crossline_patches], axis=0)

print(f"\nOriginal data shape: {seis_exploration.shape}")
print(f"Total combined patches for the CNN: {seis_exploration_patches.shape}")

print(f"\nOriginal data shape: {phi_exploration.shape}")
print(f"Total combined patches for the CNN: {phi_exploration_patches.shape}")

print(f"\nOriginal data shape: {phi_mature.shape}")
print(f"Total combined patches for the CNN: {phi_mature_patches.shape}")


In [ ]:

# --- Plotting ---
plt.figure(figsize=(8, 4))

# 1. Default plot: Appears as a VERTICAL bar
plt.subplot(1, 2, 1)
plt.title("imshow(patch)\n(Axis 0 -> Vertical)")
plt.imshow(phi_mature_patches[10,:,:,0], cmap='gray')
plt.xlabel("Depth Axis")
plt.ylabel("Crossline Axis")

# 2. Transposed plot: Appears as a HORIZONTAL bar (as expected)
plt.subplot(1, 2, 2)
plt.title("imshow(patch.T)\n(Transposed for Viewing)")
plt.imshow(phi_mature_patches[10,:,:,0].T, cmap='gray')
plt.xlabel("Crossline Axis")
plt.ylabel("Depth Axis")

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, concatenate

def build_unet_for_image_regression(input_shape=(16, 16, 1)):
    """
    Builds a U-Net model for image-to-image regression.
    """
    inputs = Input(input_shape)

    # --- Encoder Path ---
    c1 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(inputs)
    p1 = MaxPooling2D((2, 2))(c1) # 16x16 -> 8x8

    c2 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
    p2 = MaxPooling2D((2, 2))(c2) # 8x8 -> 4x4
    
    # --- Bottleneck ---
    c3 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)

    # --- Decoder Path ---
    u4 = UpSampling2D((2, 2))(c3) # 4x4 -> 8x8
    u4 = concatenate([u4, c2]) # Skip connection
    c4 = Conv2D(32, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u4)

    u5 = UpSampling2D((2, 2))(c4) # 8x8 -> 16x16
    u5 = concatenate([u5, c1]) # Skip connection
    c5 = Conv2D(16, (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u5)

    # --- Output Layer ---
    # The output is a 16x16 image with 1 channel.
    # 'linear' activation is used for regression to predict continuous values.
    outputs = Conv2D(1, (1, 1), activation='tanh')(c5)

    model = Model(inputs=[inputs], outputs=[outputs])
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mean_absolute_error'])
    
    return model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


# --- 2. Create Training and Validation Sets ---
X_train, X_val, y_train, y_val = train_test_split(
    seis_mature_patches,
    phi_mature_patches, # Use the full image patches as the target
    test_size=0.2,
    random_state=42
)
X_test = seis_exploration_patches
y_test = phi_exploration_patches

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")

# --- 3. Build and Train the U-Net Model ---
model = build_unet_for_image_regression(input_shape=(16, 16, 1))
model.summary()

In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_porosity_unet_model.keras', save_best_only=True, monitor='val_loss')

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, model_checkpoint]
)



In [ ]:
# --- 4. Evaluate and Visualize ---
print("\nEvaluating the final model on the 'exploration' test set...")
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Exploration Set Mean Absolute Error (pixel-wise): {test_mae:.4f}")

# Visualize results by comparing input, true output, and predicted output
y_pred = model.predict(X_test)
n_examples = 5
plt.figure(figsize=(12, 6))
for i in range(n_examples):
    # Plot Seismic Input
    ax = plt.subplot(3, n_examples, i + 1)
    ax.imshow(X_test[i].squeeze(), cmap='gray')
    ax.set_title("Seismic Input")
    ax.axis('off')

    # Plot True Porosity
    ax = plt.subplot(3, n_examples, i + 1 + n_examples)
    ax.imshow(y_test[i].squeeze(), cmap='viridis', vmin=0, vmax=0.3)
    ax.set_title("True Porosity")
    ax.axis('off')

    # Plot Predicted Porosity
    ax = plt.subplot(3, n_examples, i + 1 + 2 * n_examples)
    ax.imshow(y_pred[i].squeeze(), cmap='viridis', vmin=0, vmax=0.3)
    ax.set_title("Predicted Porosity")
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
y_pred.shape

In [ ]:
# 2. Transposed plot: Appears as a HORIZONTAL bar (as expected)
plt.subplot(1, 2, 2)
plt.title("imshow(patch.T)\n(Transposed for Viewing)")
plt.imshow(y_pred[50,:,:,0].T, cmap='viridis')
plt.xlabel("Crossline Axis")
plt.ylabel("Depth Axis")